# Ranking
This notebook creates the XGBoost model used to rank merchants, and shows the top 100 merchants, and the top 10 merchants by our chosen segments.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os, sys
import re
import glob
import math
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pyspark.sql.functions as F
import sklearn
import traceback
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime, timedelta
from collections import defaultdict
from multiprocessing import Manager
from xgboost import plot_importance, XGBRegressor
from pyspark.sql.window import Window
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [6]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/30 08:46:19 WARN Utils: Your hostname, tray, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/30 08:46:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/sarah/miniforge3/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/30 08:46:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Load Dataset

In [7]:
df = spark.read.parquet('.././data/curated/final_external_enriched')
df.show()

26/09/30 08:47:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+--------+------------+-------+------------------+--------------------+-----------+------------------+---------------------+---------------+--------------------+-----+------+--------------------+--------------------+--------------------+------------+---------+-------------------+------------------+--------------+------------------+-----------------------+-------------+------------------+-----------------------------+---------------------------+------------------+-------------------------+----------------------------+-------------------------+----------------------+----------+--------------+------------------+----------------+--------------+---------------+------------------+--------------+-------------+--------------+-----------------+---------------+-----------------+
|order_datetime|postcode|merchant_abn|user_id|      dollar_value|            order_id|consumer_id| fraud_probability|is_same_day_duplicate|           name|             address|state|gender|       merchant_

In [8]:
df.printSchema()

root
 |-- order_datetime: date (nullable = true)
 |-- postcode: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- consumer_id: long (nullable = true)
 |-- fraud_probability: double (nullable = true)
 |-- is_same_day_duplicate: boolean (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- category: string (nullable = true)
 |-- revenue_band: string (nullable = true)
 |-- take_rate: double (nullable = true)
 |-- category_group: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- n_transactions: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_merchant_fraud_prob: double (nullable = true)
 |-- SA

In [ ]:
# --- Merchant-level summary: take_rate + two volume metrics ---
merchant_summary = df.groupBy("merchant_abn").agg(
    F.first("take_rate").alias("take_rate"),
    F.sum("dollar_value").alias("total_dollar_value"),
    F.count("*").alias("num_transactions"),
)

print(f"Merchants summarized: {merchant_summary.count():,}")
merchant_summary.show(10)

In [ ]:
# --- Rank 1: take_rate combined with total dollar value transacted ---
take_rate_window = Window.orderBy(F.desc("take_rate"))
dollar_value_window = Window.orderBy(F.desc("total_dollar_value"))

ranked_by_dollar_value = (
    merchant_summary
    .withColumn("rank_take_rate", F.row_number().over(take_rate_window))
    .withColumn("rank_dollar_value", F.row_number().over(dollar_value_window))
    .withColumn(
        "avg_rank",
        (F.col("rank_take_rate") + F.col("rank_dollar_value")) / 2
    )
    .orderBy(F.col("avg_rank").asc())
)

print("Top merchants: take_rate + total dollar value")
ranked_by_dollar_value.select(
    "merchant_abn", "take_rate", "total_dollar_value",
    "rank_take_rate", "rank_dollar_value", "avg_rank"
).show(20, truncate=False)

In [ ]:
# --- Rank 2: take_rate combined with number of transactions ---
num_transactions_window = Window.orderBy(F.desc("num_transactions"))

ranked_by_num_transactions = (
    merchant_summary
    .withColumn("rank_take_rate", F.row_number().over(take_rate_window))
    .withColumn("rank_num_transactions", F.row_number().over(num_transactions_window))
    .withColumn(
        "avg_rank",
        (F.col("rank_take_rate") + F.col("rank_num_transactions")) / 2
    )
    .orderBy(F.col("avg_rank").asc())
)

print("Top merchants: take_rate + number of transactions")
ranked_by_num_transactions.select(
    "merchant_abn", "take_rate", "num_transactions",
    "rank_take_rate", "rank_num_transactions", "avg_rank"
).show(20, truncate=False)

In [ ]:
# --- Save both rankings ---
ranked_by_dollar_value.write.mode("overwrite").parquet("../data/curated/rankings_by_dollar_value.parquet")
ranked_by_num_transactions.write.mode("overwrite").parquet("../data/curated/rankings_by_num_transactions.parquet")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Dollar-value-based ranking
data1 = ranked_by_dollar_value.select(
    "rank_take_rate", "rank_dollar_value", "avg_rank"
).toPandas()
sc1 = axes[0].scatter(
    data1["rank_take_rate"], data1["rank_dollar_value"],
    c=data1["avg_rank"], cmap="viridis_r", s=20, alpha=0.7
)
axes[0].set_title("Take Rate Rank vs. Dollar Value Rank")
axes[0].set_xlabel("Rank by take_rate")
axes[0].set_ylabel("Rank by total_dollar_value")
fig.colorbar(sc1, ax=axes[0], label="avg_rank (lower = better)")

# Num-transactions-based ranking
data2 = ranked_by_num_transactions.select(
    "rank_take_rate", "rank_num_transactions", "avg_rank"
).toPandas()
sc2 = axes[1].scatter(
    data2["rank_take_rate"], data2["rank_num_transactions"],
    c=data2["avg_rank"], cmap="viridis_r", s=20, alpha=0.7
)
axes[1].set_title("Take Rate Rank vs. Num Transactions Rank")
axes[1].set_xlabel("Rank by take_rate")
axes[1].set_ylabel("Rank by num_transactions")
fig.colorbar(sc2, ax=axes[1], label="avg_rank (lower = better)")

plt.tight_layout()
plt.show()

In [ ]:
# Match the reference notebook's time-based split (shuffle=False, not random)
train_size = 0.7
val_size = 0.15
test_size = 1 - train_size - val_size  # 0.15

date_bounds = df.select(
    F.min("order_datetime").alias("min_date"),
    F.max("order_datetime").alias("max_date")
).collect()[0]

min_date, max_date = date_bounds["min_date"], date_bounds["max_date"]
total_days = (max_date - min_date).days

train_end = min_date + timedelta(days=int(total_days * train_size))
val_end = train_end + timedelta(days=int(total_days * (train_size + val_size)) - int(total_days * train_size))
val_end = min_date + timedelta(days=int(total_days * (train_size + val_size)))

print(f"Date range: {min_date} to {max_date} ({total_days} days)")
print(f"Train:      {min_date} to {train_end}")
print(f"Validation: {train_end} to {val_end}")
print(f"Test:       {val_end} to {max_date}")

train_df = df.filter(F.col("order_datetime") <= train_end)
val_df = df.filter((F.col("order_datetime") > train_end) & (F.col("order_datetime") <= val_end))
test_df = df.filter(F.col("order_datetime") > val_end)

print(f"Train rows:      {train_df.count():,}")
print(f"Validation rows: {val_df.count():,}")
print(f"Test rows:       {test_df.count():,}")

In [ ]:
def summarize_merchants(spark_df):
    return spark_df.groupBy("merchant_abn").agg(
        F.first("take_rate").alias("take_rate"),
        F.sum("dollar_value").alias("total_dollar_value"),
        F.count("*").alias("num_transactions"),
    )

def rank_merchants(summary_df, volume_col):
    take_rate_window = Window.orderBy(F.desc("take_rate"))
    volume_window = Window.orderBy(F.desc(volume_col))
    return (
        summary_df
        .withColumn("rank_take_rate", F.row_number().over(take_rate_window))
        .withColumn("rank_volume", F.row_number().over(volume_window))
        .withColumn("avg_rank", (F.col("rank_take_rate") + F.col("rank_volume")) / 2)
    )

train_summary = summarize_merchants(train_df)
train_ranked_dollar = rank_merchants(train_summary, "total_dollar_value").orderBy("avg_rank")
train_ranked_num_tx = rank_merchants(train_summary, "num_transactions").orderBy("avg_rank")

print("Top merchants by train-period rank (dollar value):")
train_ranked_dollar.select(
    "merchant_abn", "take_rate", "total_dollar_value", "avg_rank"
).show(10, truncate=False)

In [ ]:
test_summary = summarize_merchants(test_df).select(
    F.col("merchant_abn"),
    F.col("total_dollar_value").alias("test_total_dollar_value"),
    F.col("num_transactions").alias("test_num_transactions"),
)

validation = train_ranked_dollar.select(
    "merchant_abn", "avg_rank"
).withColumnRenamed("avg_rank", "train_avg_rank").join(
    test_summary, on="merchant_abn", how="inner"
)

test_volume_window = Window.orderBy(F.desc("test_total_dollar_value"))
validation = validation.withColumn(
    "test_rank_dollar_value", F.row_number().over(test_volume_window)
)

print(f"Merchants present in both train and test periods: {validation.count():,}")

validation_pd = validation.select(
    "merchant_abn", "train_avg_rank", "test_rank_dollar_value", "test_total_dollar_value"
).toPandas()

corr = validation_pd["train_avg_rank"].corr(
    validation_pd["test_rank_dollar_value"], method="spearman"
)
print(f"Spearman correlation (train avg_rank vs test dollar-value rank): {corr:.4f}")

plt.figure(figsize=(8, 6))
sns.scatterplot(data=validation_pd, x="train_avg_rank", y="test_rank_dollar_value", alpha=0.6)
plt.title("Train-Period Rank vs. Actual Test-Period Rank")
plt.xlabel("avg_rank (train period)")
plt.ylabel("Rank by dollar value (test period)")
plt.grid()
plt.show()